# 🎓 Ciência de Dados — Aula 14: Comitês de Modelos (*Ensembles*), Redes Neurais (*MLP*), Abordagem Híbrida e Deploy
**Professor:** Dr. Rafael Duarte  
**Data:** 12 de Novembro de 2026  
**Formato da Aula:** *Live Coding & Literate Programming* (Encerramento de 100% do Conteúdo Obrigatório do Semestre)

---

### 🗺️ Organização da Reta Final do Semestre (Aulas 14 e 15)
Hoje concluímos **100% do conteúdo técnico obrigatório** para o Projeto Semestral:
* **Bloco 1 (Primeira Parte da Aula de Hoje — Conteúdo Final):**
  * **Módulo 1:** **Sabedoria das Multidões (*Ensembles*)** — O dilema Viés vs. Variância, *Bagging* (**Random Forest**) vs. *Boosting* (**Gradient Boosting**) e Importância de Variáveis (`feature_importances_`).
  * **Módulo 2:** **Redes Neurais Artificiais (*MLP — Multi-Layer Perceptron*)** — Do neurônio matemático às camadas ocultas, função de ativação *ReLU*, *Backpropagation*, curva de aprendizado (`loss_curve_`) e o **Grande *Leaderboard* Comparativo**.
  * **Módulo 3:** **Fechamento do Pipeline CRISP-DM, Abordagem Híbrida (*Mixed Approach*) e Portabilidade entre Servidores (`joblib`)** — Como unir *Clustering* + *Supervisionado* para priorizar filas de atendimento (SLA) e como exportar o modelo treinado em um servidor para rodar em outro servidor em produção!
* **Bloco 2 (Segunda Parte da Aula de Hoje — 100% Foco no Projeto Semestral):**
  * 🚫 **Sem Exercício de Fixação Paralelo:** Todo o tempo de laboratório é dedicado exclusivamente a implementar o código de hoje **dentro do Notebook do Projeto do seu grupo**!
* **⏰ AVISO IMPORTANTE PARA A PRÓXIMA SEMANA (AULA 15 — 19/11):**
  * A próxima aula (**19/11**) **começará IMEDIATAMENTE com a Banca Presencial do Checkpoint 2 (Meta-Alvo Ótimo: 1,25 pt; Excepcional: até 1,60 pt)**.
  * **Chegada Antecipada:** Estarei em sala **mais cedo** na próxima quinta-feira para já iniciar a avaliação das equipes que chegarem antes do horário padrão. Tragam o notebook 100% executado! Após as bancas, faremos a mentoria final e guia de apresentação para a **Entrega Final em Vídeo (10 a 15 min, prazo fatal em 24/11 às 23h59)**.

---
# 🌲 MÓDULO 1: Aprendizado Supervisionado 2 — Métodos de Comitê (*Ensembles*)

## 1.1. Por que um Especialista Solitário Erra? (A Feira de Galton e o Dilema Viés vs. Variância)
Na **Aula 13**, vimos três classificadores trabalhando sozinhos: a **Regressão Logística**, o **KNN** e uma única **Árvore de Decisão (`DecisionTreeClassifier`)**.  
Qual é o problema de confiar uma decisão crítica a uma única Árvore de Decisão?
* Se a árvore for rasa demais (`max_depth=1`), ela é simplista e erra tudo (**Alto Viés / *Underfitting***).
* Se deixarmos a árvore crescer sem limites (`max_depth=None`), ela cria regras sob medida para decorar cada linha do treino: acerta $100\%$ no treino e despenca no teste (**Alta Variância / *Overfitting***).

> 🏛️ **Contexto Histórico & Etimológico (*Vox Populi* — Francis Galton, 1907):**  
> A palavra *Ensemble* vem do francês ("conjunto que atua em harmonia", do latim *insimul*, "ao mesmo tempo"). Em **1906**, o estatístico inglês **Francis Galton** analisou 787 palpites na feira de Plymouth sobre o peso de um boi. Nenhum especialista sozinho cravou o peso exato (543,4 kg), mas a **média agregada da multidão deu 542,9 kg** (erro inferior a $0,1\%$!). Quando combinamos dezenas de modelos independentes, **os erros aleatórios individuais se cancelam**!

---

## 1.2. As Duas Arquiteturas de Comitês: BAGGING vs. BOOSTING

| Característica | 🌲 **BAGGING (`RandomForestClassifier`)** | 🚀 **BOOSTING (`GradientBoostingClassifier`)** |
| :--- | :--- | :--- |
| **Como as árvores trabalham?** | **Em Paralelo (Independentes):** As 100 árvores são criadas separadamente; uma não sabe o que a outra fez. | **Em Série (Fila Indiana Sequencial):** A Árvore 2 nasce para corrigir os erros da Árvore 1; a Árvore 3 corrige a Árvore 2... |
| **Como os dados são sorteados?** | Sorteia linhas com reposição (*Bootstrap*) **e** sorteia um subconjunto de colunas em cada corte. | Usa a base para atacar os **resíduos (erros)** deixados pelas árvores anteriores. |
| **Tamanho de cada árvore** | Árvores médias/profundas (baixa restrição). | Árvores bem rasas (*Weak Learners*, ex: `max_depth=3`). |
| **O que combate melhor?** | Derruba a **Variância** (elimina o *Overfitting* da árvore única!). | Derruba o **Viés** (aprende relações complexas errando cada vez menos!). |

$$\text{Bagging (Random Forest):} \quad \hat{p}_{\text{RF}}(y_i = 1 \mid x_i) = \frac{1}{B} \sum_{b=1}^{B} T_b(x_i) \qquad \qquad \text{Boosting:} \quad F_b(x_i) = F_{b-1}(x_i) + \eta \cdot h_b(x_i)$$

<small>
📖 <b>Guia de Leitura Matemática e Notação (População vs. Amostra):</b><br>
• <b>$B$ vs. $b$ (Lê-se <i>"bê maiúsculo"</i> e <i>"bê minúsculo"</i>):</b> $B$ é o número total de árvores no comitê (parâmetro <code>n_estimators</code>, ex: $100$ árvores); $b$ é o contador da árvore individual ($b = 1, 2, \dots, B$).<br>
• <b>$T_b(x_i)$ (Lê-se <i>"tê índice bê de xis i"</i>):</b> Previsão da $b$-ésima árvore da Floresta Aleatória para o indivíduo amostral $x_i$, treinada via sorteio <i>Bootstrap</i> da nossa <b>Amostra</b> (simulando múltiplas coletas da <b>População</b>).<br>
• <b>$F_b(x_i)$ e $h_b(x_i)$:</b> $F_b$ é a previsão acumulada do Boosting no passo $b$, somando a árvore rasa corretora $h_b(x_i)$ que modela os resíduos do passo anterior.<br>
• <b>$\eta$ (Letra grega <i>Eta</i> minúscula — lê-se <i>"êta"</i>, parâmetro <code>learning_rate</code>):</b> Taxa de aprendizado (normalmente $0,05$ a $0,10$). Controla o tamanho do passo de correção para evitar que o Boosting decore ruídos.
</small>

In [ ]:
# ==============================================================================
# CÉLULA 3: IMPORTAÇÃO DE BIBLIOTECAS E CRIAÇÃO DO CENÁRIO "CONECTASUL TELECOM"
# ==============================================================================
import numpy as np                  # Biblioteca para álgebra linear, vetores e geração de números aleatórios
import pandas as pd                 # Biblioteca para manipulação de tabelas de dados (DataFrames)
import matplotlib.pyplot as plt     # Biblioteca base para criação de gráficos e figuras
import seaborn as sns               # Biblioteca de visualização estatística de alto nível
import joblib                       # Biblioteca oficial para salvar/exportar modelos treinados em disco (.joblib)

# Importando funções de preparação de dados e criação de Pipeline do Scikit-Learn:
from sklearn.model_selection import train_test_split # Divide a base em conjuntos de Treino e Teste
from sklearn.preprocessing import StandardScaler     # Padroniza variáveis na escala Z-Score (Média=0, Desvio=1)
from sklearn.pipeline import Pipeline                # Empacota pré-processamento + modelo em um único objeto portável

# Importando os algoritmos de Machine Learning (Supervisionados e Não Supervisionados):
from sklearn.linear_model import LogisticRegression          # Classificador probabilístico linear (Curva Sigmoide)
from sklearn.neighbors import KNeighborsClassifier           # Classificador geométrico por votação dos K vizinhos (KNN)
from sklearn.tree import DecisionTreeClassifier              # Árvore de Decisão única baseada em Impureza de Gini
from sklearn.ensemble import RandomForestClassifier          # Comitê em Paralelo (Bagging de centenas de árvores)
from sklearn.ensemble import GradientBoostingClassifier      # Comitê em Série (Boosting sequencial corretor de erros)
from sklearn.neural_network import MLPClassifier             # Rede Neural Artificial (Multi-Layer Perceptron)
from sklearn.cluster import KMeans                           # Agrupador Não Supervisionado por centroides (K-Médias)

# Importando as métricas de auditoria de modelos:
from sklearn.metrics import (
    confusion_matrix,       # Calcula a Matriz de Confusão (VN, FP, FN, VP)
    classification_report,  # Gera tabela completa com Precisão, Recall e F1-Score por classe
    accuracy_score,         # Calcula a Acurácia global (% total de acertos)
    precision_score,        # Calcula a Precisão: VP / (VP + FP) -> evita Alarmes Falsos
    recall_score,           # Calcula o Recall (Sensibilidade): VP / (VP + FN) -> evita Omissões Críticas
    f1_score,               # Calcula a Média Harmônica entre Precisão e Recall
    roc_auc_score           # Calcula a Área sob a Curva ROC (poder de separação probabilística de 0.5 a 1.0)
)

# Configurações visuais do Pandas e do Matplotlib para exibição limpa no projetor:
pd.set_option('display.max_rows', 25)
pd.set_option('display.width', 1000)
plt.rcParams['figure.figsize'] = (14, 5.2)
sns.set_theme(style='whitegrid')

# Fixando a semente aleatória (seed) para garantir 100% de reprodutibilidade científica:
np.random.seed(42)
n_c = 350  # Variável 'n_c': número total de contratos (linhas) da nossa amostra da operadora

# Simulando 3 tribos reais de clientes na carteira da operadora (150 Econômicos, 120 Home-Office, 80 VIPs):
# 1. Variável 'mensalidade': valor pago por mês em Reais (R$)
mensalidade = np.concatenate([
    np.random.normal(loc=90, scale=12, size=150).clip(69, 130),     # Tribo Econômica (~R$ 90)
    np.random.normal(loc=195, scale=25, size=120).clip(140, 270),   # Tribo Home-Office/Gamer (~R$ 195)
    np.random.normal(loc=920, scale=130, size=80).clip(650, 1350)   # Tribo Corporativa VIP (~R$ 920)
]).round(2)

# 2. Variável 'velocidade_mbps': velocidade de internet contratada em Megabits por segundo
velocidade_mbps = np.concatenate([
    np.random.normal(loc=200, scale=30, size=150).clip(100, 300),
    np.random.normal(loc=600, scale=60, size=120).clip(400, 800),
    np.random.normal(loc=1000, scale=80, size=80).clip(800, 1200)
]).round(0)

# 3. Variável 'chamados_6m': quantidade de chamados técnicos de manutenção abertos nos últimos 6 meses
chamados_6m = np.concatenate([
    np.random.poisson(lam=2.3, size=150).clip(0, 8),
    np.random.poisson(lam=2.1, size=120).clip(0, 7),
    np.random.poisson(lam=1.8, size=80).clip(0, 6)
])

# 4. Variável 'tempo_espera_h': tempo médio (em horas) que o cliente esperou pelo reparo técnico
tempo_espera_h = np.random.uniform(low=0.5, high=14.0, size=n_c).round(1)

# 5. Variável 'tempo_casa_meses': há quantos meses o cliente possui contrato ativo na empresa
tempo_casa_meses = np.random.randint(low=1, high=48, size=n_c)

# 6. Variável Alvo Binária 'churn_alvo' (0 = Manteve Contrato | 1 = Quebrou Contrato / Cancelou):
# Note a regra comportamental não-linear do mundo real:
# - Clientes VIP (> R$ 500) têm baixa tolerância: cancelam já a partir do 2º chamado!
# - Clientes Home-Office (R$ 140 a R$ 500) cancelam com >= 3 chamados se a espera passar de 4h.
# - Clientes Econômicos (< R$ 140) toleram mais falhas: só cancelam com >= 4 chamados (ou 3 chamados + espera > 9h).
risco_vip = (mensalidade > 500) & (chamados_6m >= 2)
risco_gam = (mensalidade >= 140) & (mensalidade <= 500) & (chamados_6m >= 3) & (tempo_espera_h > 4.0)
risco_eco = (mensalidade < 140) & ((chamados_6m >= 4) | ((chamados_6m >= 3) & (tempo_espera_h > 9.0)))
churn_base = (risco_vip | risco_gam | risco_eco).astype(int)

# Adicionando 6% de ruído estocástico natural (casos fortuitos do mundo real):
ruido_flip = np.random.binomial(n=1, p=0.06, size=n_c)
churn_alvo = np.abs(churn_base - ruido_flip)

# Consolidando todas as variáveis no DataFrame 'df_telecom' e embaralhando as linhas (.sample(frac=1.0)):
df_telecom = pd.DataFrame({
    'ID_Contrato': [f'TEL-{1001+i}' for i in range(n_c)],
    'Mensalidade_BRL': mensalidade,
    'Velocidade_Mbps': velocidade_mbps,
    'Chamados_Suporte_6M': chamados_6m,
    'Tempo_Espera_Suporte_Horas': tempo_espera_h,
    'Tempo_Casa_Meses': tempo_casa_meses,
    'Quebrou_Contrato_Churn': churn_alvo
}).sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f"✅ Base 'df_telecom' criada na Célula 3! Total: {df_telecom.shape[0]} contratos | Taxa de Churn: {df_telecom['Quebrou_Contrato_Churn'].mean()*100:.1f}%")
display(df_telecom.head(6))

In [ ]:
# ==============================================================================
# CÉLULA 4: COMPARANDO 1 ÁRVORE SOLITÁRIA (OVERFITTING) VS. RANDOM FOREST VS. BOOSTING
# (Referência Cruzada: Utilizando o DataFrame 'df_telecom' gerado na Célula 3)
# ==============================================================================

# 1. Separando as colunas explicativas (Matriz X) da coluna resposta/gabarito (Vetor y):
# 'features_base': lista com os nomes das 5 variáveis numéricas preditoras
features_base = ['Mensalidade_BRL', 'Velocidade_Mbps', 'Chamados_Suporte_6M', 'Tempo_Espera_Suporte_Horas', 'Tempo_Casa_Meses']
X = df_telecom[features_base]                # Matriz X: dados que o modelo usa para fazer a previsão
y = df_telecom['Quebrou_Contrato_Churn']     # Vetor y: gabarito binário (0 = Ficou, 1 = Cancelou)

# 2. Divisão em Treino (75% = 262 contratos) e Teste (25% = 88 contratos):
# - test_size=0.25: separa 25% dos dados como "prova final cega" (o modelo nunca vê esses dados no treino)
# - stratify=y: garante que a mesma proporção de Churn (38.3%) caia exatamente igual no Treino e no Teste!
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

# 3. Padronização de Escala Z-Score (StandardScaler) SEM VAZAMENTO DE DADOS (Anti-Data Leakage):
# - fit_transform(X_train): aprende a média e o desvio-padrão APENAS no Treino e padroniza o Treino
# - transform(X_test): aplica no Teste a mesma régua (média/desvio) aprendida no Treino!
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

# 4. Instanciando e treinando (.fit) as 3 arquiteturas baseadas em árvores:
# A) Árvore Solitária Sem Poda (max_depth=None deixa a árvore crescer até decorar 100% do treino -> Overfitting!)
arvore_sem_poda = DecisionTreeClassifier(max_depth=None, random_state=42)
arvore_sem_poda.fit(X_train_sc, y_train)

# B) Random Forest (BAGGING):
# - n_estimators=100: cria 100 árvores independentes em paralelo via sorteio Bootstrap
# - max_depth=5: limita cada árvore a 5 níveis de profundidade para garantir generalização
random_forest = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
random_forest.fit(X_train_sc, y_train)

# C) Gradient Boosting (BOOSTING):
# - n_estimators=100: cria 100 árvores sequenciais (onde cada uma corrige o resíduo da anterior)
# - learning_rate=0.08: taxa eta (tamanho do passo de correção de cada árvore)
# - max_depth=3: árvores curtas (weak learners) ideais para Boosting
grad_boosting = GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, max_depth=3, random_state=42)
grad_boosting.fit(X_train_sc, y_train)

# 5. Auditando o Overfitting (Comparando F1-Score no Treino vs. F1-Score no Teste):
for nome_arq, modelo_treinado in [
    ('1 Árvore Solitária Sem Poda (Sofre Overfitting!)', arvore_sem_poda),
    ('Random Forest - 100 Árvores em Paralelo (Bagging)', random_forest),
    ('Gradient Boosting - 100 Árvores em Série (Boosting)', grad_boosting)
]:
    f1_no_treino = f1_score(y_train, modelo_treinado.predict(X_train_sc))
    f1_no_teste  = f1_score(y_test, modelo_treinado.predict(X_test_sc))
    gap_overfit  = f1_no_treino - f1_no_teste # Se o Gap for muito alto (> 0.15), o modelo decorou o treino!
    print(f"🔹 {nome_arq:52s} -> F1 Treino: {f1_no_treino:.4f} | F1 Teste: {f1_no_teste:.4f} (Gap: {gap_overfit:+.4f})")

# 6. Extraindo a Importância das Variáveis (.feature_importances_) da Random Forest:
# O atributo 'feature_importances_' mede quanto cada coluna reduziu a Impureza de Gini na média das 100 árvores!
df_imp = pd.DataFrame({
    'Variavel': features_base,
    'Importancia_Pct': (random_forest.feature_importances_ * 100).round(2)
}).sort_values(by='Importancia_Pct', ascending=True)

plt.figure(figsize=(9, 3.8))
plt.barh(df_imp['Variavel'], df_imp['Importancia_Pct'], color='teal', edgecolor='black')
plt.title('Importância das Variáveis na Random Forest (Quem mais influencia a decisão do comitê?)')
plt.xlabel('Importância Relativa na Redução da Impureza de Gini (%)')
plt.tight_layout()
plt.show()

---
# 🧠 MÓDULO 2: Redes Neurais Artificiais (*Multi-Layer Perceptron — MLP*)

## 2.1. Desmistificando a Caixa-Preta: O Que é um Neurônio Artificial?
Um neurônio artificial nada mais é do que a **Regressão Linear/Logística empilhada em camadas**:
* **1943 (McCulloch & Pitts):** Primeiro modelo lógico de neurônio.
* **1958 (Perceptron de Frank Rosenblatt):** Primeira máquina física que aprendia pesos sinápticos $w_j$.
* **1986 (Rumelhart, Hinton & Williams):** Algoritmo de **Retropropagação do Erro (*Backpropagation*)** para treinar múltiplas camadas ocultas (que rendeu a Geoffrey Hinton o **Prêmio Nobel de Física de 2024**).

$$z_k = \sum_{j=1}^{m} w_{jk} x_{ij} + b_k \quad \Longrightarrow \quad a_k = \text{ReLU}(z_k) = \max(0, z_k)$$

<small>
📖 <b>Guia de Leitura Matemática e Notação:</b><br>
• <b>$w_{jk}$ (Lê-se <i>"dáblio índice jota, ká"</i> — Peso Sináptico):</b> Equivalente ao peso $b_j$ (estimador de $\beta_j$) da Regressão! Mede a força da conexão entre a variável de entrada $j$ e o neurônio $k$.<br>
• <b>$b_k$ (Lê-se <i>"bê índice ká"</i> — <i>Bias</i>):</b> Constante de ativação do neurônio $k$ (equivalente ao intercepto $b_0$).<br>
• <b>$a_k = \max(0, z_k)$ (Função de Ativação ReLU — <i>Rectified Linear Unit</i>):</b> Se a soma ponderada $z_k$ for negativa, o neurônio zera o sinal ($0$); se for positiva, deixa passar ($z_k$). Na camada final de saída, a rede aplica a <b>Função Sigmoide $\sigma(z)$</b> para entregar a probabilidade entre $0$ e $1$.
</small>

In [ ]:
# ==============================================================================
# CÉLULA 6: TREINANDO A REDE NEURAL ARTIFICIAL (MLP) E PLOTANDO A CURVA DE PERDA
# (Referência Cruzada: Utilizando 'X_train_sc' e 'y_train' padronizados na Célula 4)
# ==============================================================================

# Instanciando o Perceptron Multicamadas (MLPClassifier):
# - hidden_layer_sizes=(32, 16): define 2 camadas ocultas (a 1ª com 32 neurônios e a 2ª com 16 neurônios)
# - activation='relu': aplica a função não-linear ReLU max(0, z) em todos os neurônios ocultos
# - solver='adam': algoritmo otimizador que ajusta os pesos w_jk via Gradiente Descendente (Backpropagation)
# - max_iter=350: número máximo de épocas (ciclos completos de ida e volta pelos dados de treino)
rede_neural_mlp = MLPClassifier(
    hidden_layer_sizes=(32, 16),
    activation='relu',
    solver='adam',
    max_iter=350,
    random_state=42
)

# O método .fit() executa o Forward Pass e o Backpropagation época por época:
rede_neural_mlp.fit(X_train_sc, y_train)

# O atributo .loss_curve_ guarda o valor do erro (Log-Loss) da rede ao final de cada época:
plt.figure(figsize=(9, 3.8))
plt.plot(rede_neural_mlp.loss_curve_, color='purple', lw=2.5, label='Erro da Rede Neural (Log-Loss no Treino)')
plt.title(f'Curva de Aprendizado da Rede Neural MLP (Convergiu após {rede_neural_mlp.n_iter_} épocas!)')
plt.xlabel('Épocas de Treinamento (Ciclos de Backpropagation)')
plt.ylabel('Erro Log-Loss (Quanto menor, mais a rede aprendeu)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ==============================================================================
# CÉLULA 7: O GRANDE LEADERBOARD — COMPARANDO OS 6 ALGORITMOS NA MESMA BASE
# (Referência Cruzada: Utilizando 'X_train_sc', 'X_test_sc', 'y_train', 'y_test' da Célula 4)
# ==============================================================================

# Dicionário contendo os 6 classificadores (3 Clássicos da Aula 13 + 2 Ensembles e 1 Rede Neural da Aula 14):
bateria_6_modelos = {
    '1. Regressão Logística (Clássico - Aula 13)': LogisticRegression(random_state=42),
    '2. KNN - 5 Vizinhos (Clássico - Aula 13)': KNeighborsClassifier(n_neighbors=5),
    '3. Árvore de Decisão Podada (Clássico - Aula 13)': DecisionTreeClassifier(max_depth=4, random_state=42),
    '4. Random Forest - Bagging (Ensemble - Aula 14)': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42),
    '5. Gradient Boosting (Ensemble - Aula 14)': GradientBoostingClassifier(n_estimators=100, learning_rate=0.08, max_depth=3, random_state=42),
    '6. Rede Neural MLP 32x16 (Deep Tabular - Aula 14)': MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu', solver='adam', max_iter=350, random_state=42)
}

linhas_leaderboard = [] # Lista vazia onde guardaremos o dicionário de métricas de cada modelo

for nome_m, mod_obj in bateria_6_modelos.items():
    # 1. Treina o modelo nos dados de treino padronizados:
    mod_obj.fit(X_train_sc, y_train)
    
    # 2. Gera a previsão de classe (0 ou 1) e a probabilidade contínua (0.0 a 1.0) nos dados de TESTE:
    pred_te = mod_obj.predict(X_test_sc)
    prob_te = mod_obj.predict_proba(X_test_sc)[:, 1] # Pega a coluna 1 (probabilidade de Churn = 1)
    
    # 3. Calcula o F1-Score no Treino e no Teste para auditar se houve Overfitting:
    f1_tr = f1_score(y_train, mod_obj.predict(X_train_sc))
    f1_te = f1_score(y_test, pred_te)
    
    # 4. Armazena todas as métricas na lista:
    linhas_leaderboard.append({
        'Modelo Supervisionado': nome_m,
        'Acurácia (Teste)': round(accuracy_score(y_test, pred_te), 4),
        'Precisão (Teste)': round(precision_score(y_test, pred_te, zero_division=0), 4),
        'Recall (Teste)': round(recall_score(y_test, pred_te), 4),
        'F1-Score (Teste)': round(f1_te, 4),
        'F1-Score (Treino)': round(f1_tr, 4),
        'Gap Overfitting (Treino-Teste)': round(f1_tr - f1_te, 4),
        'ROC-AUC': round(roc_auc_score(y_test, prob_te), 4)
    })

# Converte a lista em DataFrame e ordena do maior F1-Score no Teste para o menor:
df_leaderboard = pd.DataFrame(linhas_leaderboard).sort_values(by='F1-Score (Teste)', ascending=False).reset_index(drop=True)
print("🏆 LEADERBOARD OFICIAL DE MODELOS (TABELA EXIGIDA NO CHECKPOINT 2 E NO VÍDEO FINAL):")
display(df_leaderboard)

---
# 🔗 MÓDULO 3: Fechamento do Pipeline — Abordagem Híbrida (*Mixed Approach*) e Portabilidade entre Servidores (`joblib`)

## 3.1. A Abordagem Híbrida (*Mixed Approach*): Unindo Clustering + Classificação
Como vimos no nosso estudo de caso da **ConectaSul Telecom**, diferentes tribos de clientes têm **curvas de tolerância a falhas completamente distintas**:
* **Passo 1 (Não Supervisionado — `KMeans`):** Agrupa os clientes pelo porte do contrato (`Mensalidade_BRL` e `Velocidade_Mbps`), descobrindo a **Tribo / Persona** sem olhar para o *Churn*.
* **Passo 2 (Supervisionado + *Cluster-as-a-Feature*):** O rótulo do cluster descoberto pelo K-Means entra como variável preditora nos classificadores, alimentando o **Motor Inteligente de Priorização de Fila (SLA)**!

---

## 3.2. Como Portar o Conhecimento? (Treinar no Servidor A e Rodar em Produção no Servidor B!)
Uma dúvida fundamental de arquitetura de software e MLOps:  
> *"Depois que eu treinei o meu `StandardScaler` e o meu `Modelo Campeão` no servidor de treinamento (ou no meu notebook), como eu levo essa inteligência pronta para rodar em outro servidor (ex: uma API FastAPI/Flask na AWS ou um sistema de atendimento) sem precisar levar a base de treino junto e sem retreinar tudo do zero?"*

A resposta profissional tem **duas peças**:
1. **`sklearn.pipeline.Pipeline`:** Empacota o `StandardScaler()` e o `Modelo` dentro de **um único objeto unificado**. Assim, no servidor de produção, você passa os dados brutos (em Reais, Mbps, etc.) e o próprio pacote padroniza com a média/desvio do treino e já devolve a previsão!
2. **`joblib.dump(objeto, 'arquivo.joblib')` e `joblib.load('arquivo.joblib')`:** Serializa (congela em binário) todos os pesos matemáticos aprendidos ($w_{jk}$, árvores, médias $\mu_j$ e desvios $\sigma_j$) em um arquivo leve de poucos Kilobytes que você copia para qualquer servidor do mundo!

In [ ]:
# ==============================================================================
# CÉLULA 9: PIPELINE HÍBRIDO (CLUSTERING K-MEANS + CURVAS DE TOLERÂNCIA + MOTOR SLA)
# (Referência Cruzada: Utilizando 'X_train', 'X_test', 'y_train' criados na Célula 4)
# ==============================================================================

# PASSO 1 (NÃO SUPERVISIONADO): Descobrindo o Cluster de Perfil do Contrato via K-Means
scaler_perfil = StandardScaler()
# Ajusta a régua apenas nas duas colunas de perfil contratual do Treino:
X_perfil_train = scaler_perfil.fit_transform(X_train[['Mensalidade_BRL', 'Velocidade_Mbps']])
X_perfil_test  = scaler_perfil.transform(X_test[['Mensalidade_BRL', 'Velocidade_Mbps']])

# Treina o K-Means com K = 3 grupos nos dados de treino:
km_hibrido = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_perfil_train)

# Batizando os 3 clusters automaticamente pela ordem crescente de mensalidade média:
df_temp_tr = X_train.copy()
df_temp_tr['Cluster_Bruto'] = km_hibrido.labels_
df_temp_tr['Churn'] = y_train
ordem_tribos = df_temp_tr.groupby('Cluster_Bruto')['Mensalidade_BRL'].mean().sort_values().index
mapa_tribos = {
    ordem_tribos[0]: '1_Economico_Residencial',
    ordem_tribos[1]: '2_Gamer_HomeOffice',
    ordem_tribos[2]: '3_Corporativo_VIP'
}
df_temp_tr['Tribo_Cliente'] = df_temp_tr['Cluster_Bruto'].map(mapa_tribos)

# PASSO 2: Plotando a Curva de Tolerância a Chamados Técnicos dentro de cada Cluster!
plt.figure(figsize=(11, 4.5))
cores_t = {'1_Economico_Residencial': 'seagreen', '2_Gamer_HomeOffice': 'darkorange', '3_Corporativo_VIP': 'crimson'}
grid_chamados = pd.DataFrame({'Chamados_Suporte_6M': np.linspace(0, 7, 100)})

for nome_tribo, cor in cores_t.items():
    fatia = df_temp_tr[df_temp_tr['Tribo_Cliente'] == nome_tribo]
    mod_fatia = LogisticRegression().fit(fatia[['Chamados_Suporte_6M']], fatia['Churn'])
    curva_prob = mod_fatia.predict_proba(grid_chamados)[:, 1] * 100
    plt.plot(grid_chamados['Chamados_Suporte_6M'], curva_prob, lw=3, color=cor, label=f'Tribo: {nome_tribo}')

plt.axhline(50, color='black', linestyle='--', alpha=0.7, label='Gatilho Crítico de Risco (50% de Chance de Quebra)')
plt.title('Mixed Approach: Curva de Tolerância a Chamados de Manutenção por Cluster de Cliente')
plt.xlabel('Quantidade de Chamados Técnicos Abertos no Semestre')
plt.ylabel('Probabilidade Estimada de Quebra de Contrato (%)')
plt.xticks(range(0, 8))
plt.legend()
plt.tight_layout()
plt.show()

# PASSO 3 (CLUSTER-AS-A-FEATURE): Adicionando o Cluster descoberto pelo K-Means como nova coluna preditora!
X_train_hib = X_train.copy()
X_test_hib  = X_test.copy()
X_train_hib['Cluster_ID'] = km_hibrido.predict(X_perfil_train)
X_test_hib['Cluster_ID']  = km_hibrido.predict(X_perfil_test)

# Criando o Pipeline Híbrido (StandardScaler + Rede Neural MLP Campeã) e treinando:
pipeline_hibrido = Pipeline([
    ('padronizador_zscore', StandardScaler()),
    ('rede_neural_campea', MLPClassifier(hidden_layer_sizes=(32, 16), activation='relu', solver='adam', max_iter=350, random_state=42))
])
pipeline_hibrido.fit(X_train_hib, y_train)

# Simulando o Motor Inteligente de Priorização de Fila (SLA) para os 6 primeiros contratos do Teste:
df_fila_sla = X_test.iloc[:6].copy()
df_fila_sla['Tribo_KMeans'] = pd.Series(km_hibrido.predict(X_perfil_test[:6]), index=df_fila_sla.index).map(mapa_tribos)
probs_sla = pipeline_hibrido.predict_proba(X_test_hib.iloc[:6])[:, 1]
df_fila_sla['Prob_Quebra_Contrato (%)'] = (probs_sla * 100).round(1)

def definir_sla(row):
    if row['Prob_Quebra_Contrato (%)'] >= 70 or (row['Tribo_KMeans'] == '3_Corporativo_VIP' and row['Prob_Quebra_Contrato (%)'] >= 40):
        return '🔴 PRIORIDADE 1: SLA 2h (Equipe Sênior + Alerta Retenção)'
    elif row['Prob_Quebra_Contrato (%)'] >= 35:
        return '🟡 PRIORIDADE 2: SLA 12h (Fila Rápida)'
    else:
        return '🟢 PRIORIDADE 3: SLA 48h (Atendimento Padrão)'

df_fila_sla['Acao_Prescritiva_Fila_SLA'] = df_fila_sla.apply(definir_sla, axis=1)
print("🚀 MOTOR INTELIGENTE DE PRIORIZAÇÃO DE CHAMADOS EM PRODUÇÃO (OUTPUT PRESCRITIVO):")
display(df_fila_sla[['Mensalidade_BRL', 'Chamados_Suporte_6M', 'Tempo_Espera_Suporte_Horas', 'Tribo_KMeans', 'Prob_Quebra_Contrato (%)', 'Acao_Prescritiva_Fila_SLA']])

In [ ]:
# ==============================================================================
# CÉLULA 10: PORTABILIDADE DE CONHECIMENTO — TREINAR NO SERVIDOR 'A' E RODAR NO SERVIDOR 'B'
# (Referência Cruzada: Exportando os objetos treinados na Célula 9 para um arquivo binário)
# ==============================================================================

# ------------------------------------------------------------------------------
# ETAPA A (NO SERVIDOR DE TREINAMENTO / NOTEBOOK DO CIENTISTA DE DADOS):
# Empacotamos tudo o que o sistema de produção precisa em um dicionário de artefatos
# e salvamos em um único arquivo físico '.joblib' no disco!
# ------------------------------------------------------------------------------
pacote_conhecimento_ia = {
    'Versao_Modelo': 'v1.0_ConectaSul_2026',
    'Colunas_Perfil_KMeans': ['Mensalidade_BRL', 'Velocidade_Mbps'],
    'Colunas_Preditoras_X': list(X_train_hib.columns),
    'Scaler_Perfil_KMeans': scaler_perfil,       # Guarda as médias e desvios do K-Means
    'Modelo_KMeans_Tribos': km_hibrido,          # Guarda os centroides aprendidos das 3 tribos
    'Mapa_Nomes_Tribos': mapa_tribos,            # Dicionário que traduz 0, 1, 2 para o nome da Persona
    'Pipeline_Supervisionado': pipeline_hibrido  # Contém o StandardScaler + Rede Neural MLP já treinados!
}

nome_arquivo_exportado = 'artefato_modelo_conectasul_v1.joblib'
joblib.dump(pacote_conhecimento_ia, nome_arquivo_exportado)
print(f"💾 [SERVIDOR DE TREINO] Conhecimento salvo com sucesso no arquivo '{nome_arquivo_exportado}'!\n")

# ------------------------------------------------------------------------------
# ETAPA B (EM OUTRO SERVIDOR DE PRODUÇÃO / API WEB / SISTEMA DE CHAMADOS):
# Imagine que este código abaixo está rodando em outro computador na nuvem (AWS/Docker),
# SEM acesso à base de dados antiga. Ele apenas carrega o arquivo '.joblib' e prevê!
# ------------------------------------------------------------------------------
artefato_carregado = joblib.load(nome_arquivo_exportado)
print(f"☁️ [SERVIDOR DE PRODUÇÃO] Artefato '{artefato_carregado['Versao_Modelo']}' carregado na memória RAM!")

# Chegaram 2 chamados novos agora na API do Sistema de Atendimento:
novos_chamados_api = pd.DataFrame({
    'Mensalidade_BRL': [980.00, 89.90],          # Cliente 1 é VIP (R$ 980); Cliente 2 é Econômico (R$ 89,90)
    'Velocidade_Mbps': [1000.0, 200.0],
    'Chamados_Suporte_6M': [2, 2],               # Ambos estão abrindo o 2º chamado no semestre!
    'Tempo_Espera_Suporte_Horas': [5.5, 5.5],
    'Tempo_Casa_Meses': [12, 12]
}, index=['Chamado_Novo_01 (Empresa VIP)', 'Chamado_Novo_02 (Residencial Econômico)'])

# 1. O Servidor B descobre a Tribo usando o K-Means carregado do arquivo:
matriz_p_novo = artefato_carregado['Scaler_Perfil_KMeans'].transform(novos_chamados_api[artefato_carregado['Colunas_Perfil_KMeans']])
cluster_id_novo = artefato_carregado['Modelo_KMeans_Tribos'].predict(matriz_p_novo)
novos_chamados_api['Cluster_ID'] = cluster_id_novo

# 2. O Servidor B calcula a Probabilidade de Quebra de Contrato usando o Pipeline carregado:
probs_novos = artefato_carregado['Pipeline_Supervisionado'].predict_proba(novos_chamados_api[artefato_carregado['Colunas_Preditoras_X']])[:, 1]
novos_chamados_api['Tribo_Identificada'] = pd.Series(cluster_id_novo, index=novos_chamados_api.index).map(artefato_carregado['Mapa_Nomes_Tribos'])
novos_chamados_api['Risco_Churn_Previsto (%)'] = (probs_novos * 100).round(1)

print("\n🎯 PREVISÃO EM TEMPO REAL NO SERVIDOR DE PRODUÇÃO (Note que ambos têm 2 chamados, mas riscos opostos!):")
display(novos_chamados_api[['Mensalidade_BRL', 'Chamados_Suporte_6M', 'Tribo_Identificada', 'Risco_Churn_Previsto (%)']])

---
# ☕ INTERVALO (15 Minutos)
Na volta do intervalo: **Desenvolvimento Direto do Projeto Semestral do seu grupo rumo ao Checkpoint 2!**

---
# 🚀 MÓDULO 4: Diretrizes Oficiais do Checkpoint 2 (19/11) e da Entrega Final (24/11)

> ⏰ **AVISO CRÍTICO PARA A PRÓXIMA AULA (AULA 15 — 19/11):**  
> Na semana que vem (**19/11**), **a aula começará IMEDIATAMENTE com a Banca Presencial do Checkpoint 2 (mesa em mesa)**.  
> **O professor chegará mais cedo à sala de aula** para já iniciar as bancas com os grupos que estiverem prontos antes do horário padrão. Tragam o notebook do grupo 100% executado!

### 📊 4.1. Tabela de Critérios — CHECKPOINT 2 (Meta-Alvo Ótimo: 1,25 Ponto)
* **Regra de Pontuação por Nível:** Se a equipe entregar todos os itens no nível **Ótimo**, garante **100% da nota da etapa (1,25 pt)**. Entregas em nível **Excepcional** valem $+25\%$ no critério (arredondado em $0,05$), **Adequado** vale $60\%$ do alvo, **Insuficiente** vale $30\%$ do alvo e **Não Atendido / Zerado** não pontua ($0,00$).

| Critério Avaliado | 🌟 Excepcional (+25%) | ✅ Ótimo (Alvo 100%) | ⚠️ Adequado (60%) | 🔴 Insuf. (30%) / Zerado (0%) |
| :--- | :---: | :---: | :---: | :---: |
| **1. Alvo Binário, Stratify e Escala** | **0,30** | **0,25** | **0,15** | **0,10 / 0,00** |
| **2. Clássicos, Ensembles e MLP** | **0,45** | **0,35** | **0,20** | **0,10 / 0,00** |
| **3. Matriz de Confusão e Métricas** | **0,40** | **0,30** | **0,20** | **0,10 / 0,00** |
| **4. Defesa Técnica na Bancada** | **0,45** | **0,35** | **0,20** | **0,10 / 0,00** |
| **SOMA DA ETAPA (CP2)** | **1,60** | **1,25** | **0,75** | **0,40 / 0,00** |

---

### 📊 4.2. Tabela de Critérios — APRESENTAÇÃO E ENTREGA FINAL (Prazo Fatal: 24/11 às 23h59 — Meta-Alvo Ótimo: 2,50 Pontos)
* **Pacote de Entrega Final:** **(1)** Notebook Definitivo (`.ipynb`) + Base de Dados; **(2)** Vídeo Pitch Executivo (**10 a 15 minutos** — YouTube não-listado/Drive/MP4 com câmera e voz de todos os integrantes); **(3)** Avaliação por Pares (Soma Zero: $5 \times (N-1)$ pontos).

| Critério Avaliado | 🌟 Excepcional (+25%) | ✅ Ótimo (Alvo 100%) | ⚠️ Adequado (60%) | 🔴 Insuf. (30%) / Zerado (0%) |
| :--- | :---: | :---: | :---: | :---: |
| **1. Engenharia do Notebook Final** | **0,75** | **0,60** | **0,35** | **0,20 / 0,00** |
| **2. Benchmark e Modelo Campeão** | **0,80** | **0,65** | **0,40** | **0,20 / 0,00** |
| **3. Storytelling, ROI e LGPD** | **0,80** | **0,65** | **0,40** | **0,20 / 0,00** |
| **4. Defesa Técnica em Vídeo** | **0,75** | **0,60** | **0,35** | **0,20 / 0,00** |
| **SOMA DA ETAPA FINAL** | **3,10** | **2,50** | **1,50** | **0,80 / 0,00** |

In [ ]:
# ==============================================================================
# ESPAÇO DE DESENVOLVIMENTO DO PROJETO DO GRUPO (RODE SEUS DADOS AQUI!)
# ==============================================================================
# Use o código das Células 4, 6, 7, 9 e 10 acima como molde para aplicar na base do seu grupo!